In [ ]:
# 150mm CENTER-CROP preprocess (discussion lever: Tucker 11th/Tom 53rd use 150mm crop -> mm-level detail).
# Crop central 150mm via DICOM PixelSpacing, window, resize 384px. K=8/series. Push to PRIVATE HF.
import os, glob, time, numpy as np, pandas as pd, pydicom, cv2
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, create_repo
K=8; IMG=384; CROP_MM=150.0; SHARD_STUDIES=30
HF_TOKEN=UserSecretsClient().get_secret('HF_TOKEN')
api=HfApi(token=HF_TOKEN)
USER=api.whoami()['name']; REPO=f'{USER}/rsna-knee-384-crop'
print('HF user:', USER, '| repo:', REPO)
create_repo(REPO, repo_type='dataset', private=True, exist_ok=True, token=HF_TOKEN)
base=None
for r in sorted(glob.glob('/kaggle/input/*')):
    h=glob.glob(r+'/**/train_series.csv', recursive=True)
    if h: base=os.path.dirname(h[0]); break
print('base', base)
tr=pd.read_csv(os.path.join(base,'train.csv')); ts=pd.read_csv(os.path.join(base,'train_series.csv'))
studies=list(tr['StudyInstanceUID']); print('studies', len(studies))
PL={'Sagittal':0,'Coronal':1,'Axial':2}

In [ ]:
def sampled(sd,k):
    fs=sorted(glob.glob(sd+'/*.dcm')); n=len(fs)
    if n==0: return []
    if n<=k: return fs
    lo,hi=int(n*.12),int(n*.88)
    if hi<=lo: lo,hi=0,n
    return [fs[int(round(x))] for x in np.linspace(lo,hi-1,k)]
def slc(p):
    try:
        d=pydicom.dcmread(p); a=d.pixel_array.astype(np.float32)
        a=a*float(getattr(d,'RescaleSlope',1) or 1)+float(getattr(d,'RescaleIntercept',0) or 0)
        ps=getattr(d,'PixelSpacing',None)  # [row_mm, col_mm]
        if ps is not None:
            try:
                psr=float(ps[0]); psc=float(ps[1]); H,W=a.shape
                cr=min(int(round(CROP_MM/psr)),H); cc=min(int(round(CROP_MM/psc)),W)
                r0=max(0,(H-cr)//2); c0=max(0,(W-cc)//2)
                a=a[r0:r0+cr, c0:c0+cc]  # central 150mm
            except Exception: pass
        lo,hi=np.percentile(a,0.5),np.percentile(a,99.5); a=np.clip((a-lo)/max(hi-lo,1e-6),0,1)
        return (cv2.resize(a,(IMG,IMG),interpolation=cv2.INTER_AREA)*255).astype(np.uint8)
    except Exception: return None
t0=time.time(); imgs=[]; meta=[]; sids=[]; shard=0
def flush():
    global imgs,meta,sids,shard
    if not imgs: return
    fn=f'shard_{shard:03d}.npz'
    np.savez_compressed(fn, img=np.stack(imgs), meta=np.array(meta), study_ids=np.array(sids))
    api.upload_file(path_or_fileobj=fn, path_in_repo=fn, repo_id=REPO, repo_type='dataset', token=HF_TOKEN)
    os.remove(fn); print(f'  pushed {fn} ({shard+1}) ~{(shard+1)*SHARD_STUDIES} studies {time.time()-t0:.0f}s', flush=True)
    imgs=[]; meta=[]; sids=[]; shard+=1
for si,study in enumerate(studies):
    for _,sr in ts[ts['StudyInstanceUID']==study].iterrows():
        sd=os.path.join(base,'train_series',study,sr['SeriesInstanceUID'])
        if not os.path.isdir(sd): continue
        pid=PL.get(sr['Anatomical_Plane'],0); fb=int(sr['Fluid_Sensitive'])
        for p in sampled(sd,K):
            im=slc(p)
            if im is not None: imgs.append(im); meta.append((si,pid,fb))
    sids.append(study)
    if (si+1)%SHARD_STUDIES==0: flush()
flush()
pd.DataFrame({'si':range(len(studies)),'StudyInstanceUID':studies}).to_csv('study_index.csv',index=False)
api.upload_file(path_or_fileobj='study_index.csv', path_in_repo='study_index.csv', repo_id=REPO, repo_type='dataset', token=HF_TOKEN)
print('DONE 150mm-crop 384px -> HF', REPO, f'{time.time()-t0:.0f}s', 'shards', shard)